In [ ]:
# ======================================================
# Notebook: 6D Hyperparameter Optimisation (Black-box)
# Inputs: (30,6) | Output: (30,)
# Goal: maximise performance
# ======================================================

import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from scipy.stats import norm

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (30,6)
y = np.load("/mnt/data/initial_outputs.npy")     # (30,)

# GP surrogate
kernel = ConstantKernel(1.0) * Matern(nu=2.5) + WhiteKernel()
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=5)
gp.fit(X, y)

# Expected Improvement
def expected_improvement(X_cand, X_sample, model, xi=0.01):
    mu, sigma = model.predict(X_cand, return_std=True)
    best = np.max(model.predict(X_sample))
    mu = mu.reshape(-1,1)
    sigma = sigma.reshape(-1,1)

    imp = mu - best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma == 0] = 0
    return ei.ravel()

# Candidate sampling
bounds = [(X[:,i].min(), X[:,i].max()) for i in range(6)]
n_candidates = 5000
X_grid = np.column_stack([
    np.random.uniform(b[0], b[1], n_candidates) for b in bounds
])

# Acquisition
ei = expected_improvement(X_grid, X, gp)

# Next (10,6)
top_idx = np.argsort(ei)[-10:]
next_points = X_grid[top_idx]

print("Next (10,6) hyperparameter candidates:")
print(next_points)